# Aula 16 - Notebook: Circuitos Eulerianos e Inspeção Autônoma de Tubulações

Neste notebook modelamos a malha física de manutenção e implementamos o **Algoritmo de Hierholzer** para gerar a rota ótima de inspeção do robô industrial.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII pura."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

from collections import defaultdict
from typing import List, Tuple

class GrafoInspecaoEuleriano:
    def __init__(self):
        self.adj = defaultdict(list)
        
    def adicionar_duto(self, u: str, v: str, id_duto: str):
        self.adj[u].append((v, id_duto))
        self.adj[v].append((u, id_duto))
        
    def verificar_euleriano(self) -> Tuple[bool, List[str]]:
        impares = [v for v, viz in self.adj.items() if len(viz) % 2 != 0]
        return len(impares) == 0, impares

    def calcular_circuito_hierholzer(self, inicio: str) -> List[str]:
        adj_copia = {u: list(viz) for u, viz in self.adj.items()}
        pilha = [inicio]
        circuito = []
        while pilha:
            u = pilha[-1]
            if adj_copia[u]:
                v, id_e = adj_copia[u].pop()
                adj_copia[v].remove((u, id_e))
                pilha.append(v)
            else:
                circuito.append(pilha.pop())
        circuito.reverse()
        return circuito

# Grafo com todos os vértices de grau par (grau 2 ou 4)
g_insp = GrafoInspecaoEuleriano()
g_insp.adicionar_duto("Base", "TK301", "d1")
g_insp.adicionar_duto("TK301", "MAN101", "d2")
g_insp.adicionar_duto("MAN101", "R101", "d3")
g_insp.adicionar_duto("R101", "TK303", "d4")
g_insp.adicionar_duto("TK303", "GRAN201", "d5")
g_insp.adicionar_duto("GRAN201", "Base", "d6")
# Anel interno balanceado
g_insp.adicionar_duto("TK301", "R101", "d7")
g_insp.adicionar_duto("R101", "MAN101", "d8")
g_insp.adicionar_duto("MAN101", "TK301", "d9")

eul, imp = g_insp.verificar_euleriano()
print(f"Grafo é Euleriano: {eul} | Vértices Ímpares: {imp}")
rota_robo = g_insp.calcular_circuito_hierholzer("Base")
print("Circuito Euleriano de Inspeção:", " -> ".join(rota_robo))
assert eul is True


Grafo é Euleriano: True | Vértices Ímpares: []
Circuito Euleriano de Inspeção: Base -> GRAN201 -> TK303 -> R101 -> MAN101 -> TK301 -> R101 -> MAN101 -> TK301 -> Base
